In [1]:
import torch
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# --------------------------------------------------------------------------
# --- 1. 모델 로드 ---
# --------------------------------------------------------------------------
# Hugging Face 토큰이 필요한 경우 미리 로그인합니다. (선택 사항)
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")

print("LLM (Llama 3.1)을 로딩합니다...")
try:
    # 사용할 GPU 장치 번호 설정
    device_num = 0
    model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    
    # 텍스트 생성 파이프라인 설정
    llm_pipe = pipeline(
        "text-generation",
        model=model_id,
        model_kwargs={"torch_dtype": torch.bfloat16}, # bfloat16으로 메모리 효율성 증대
        device_map=f"cuda:{device_num}", # 특정 GPU에 할당
    )
    print("✅ LLM 로딩이 완료되었습니다.")
except Exception as e:
    print(f"❌ LLM 모델 로딩 중 오류가 발생했습니다: {e}")
    print("GPU 메모리가 부족하거나, 인터넷 연결, Hugging Face 토큰을 확인해주세요.")
    exit()

# --------------------------------------------------------------------------
# --- 2. 경로 설정 ---
# --------------------------------------------------------------------------
# 입력 및 출력 디렉토리 경로
base_dir = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/"
output_dir = os.path.join(base_dir, "preprocess/caption")

# 디렉토리가 없으면 생성
os.makedirs(output_dir, exist_ok=True)

# 입출력 파일 전체 경로
input_csv_path = os.path.join(base_dir, "dev_test.csv")
output_csv_path = os.path.join(output_dir, "test_preprocessed_for_clip.csv")

print(f"입력 파일: {input_csv_path}")
print(f"출력 파일: {output_csv_path}")

# --------------------------------------------------------------------------
# --- 3. 프롬프트 템플릿 정의 ---
# --------------------------------------------------------------------------
prompt_template_for_clip = """
You are an expert AI assistant specializing in rephrasing text for visual-language models like CLIP.
Your goal is to convert multiple-choice options into clear, descriptive, and affirmative sentences that can be easily compared to an image.

**Instructions:**
1.  **Create Complete Sentences:** Transform each option from a word or phrase into a full, descriptive sentence.
2.  **Always Be Affirmative:** Create positive, declarative statements. For example, if the option is "Pizza", rephrase it as "There is a pizza in the photo."
3.  **Ignore Question Logic:** This is crucial. Completely ignore the framing of the question (e.g., "Which is not...", "What is the purpose..."). Your ONLY job is to rephrase the options (A, B, C, D) into standalone, positive descriptions.
4.  **Consistent Structure:** Use a consistent and descriptive sentence structure.
5.  **JSON Output:** Provide your response ONLY as a single JSON object with keys "A", "B", "C", and "D".

### Example 1
Question: Which of the following foods is NOT present in the image?
A: Pizza
B: Blueberries
C: Salmon
D: Avocado

### Your JSON Output 1
{{
  "A": "There is a pizza in the photo.",
  "B": "There are blueberries in the photo.",
  "C": "There is a salmon in the photo.",
  "D": "There is an avocado in the photo."
}}

### Example 2
Question: What might be the purpose of the person's workout in the image?
A: Building muscle and strength
B: Practicing for a marathon
C: Training for a cycling race
D: Preparing for a swimming competition

### Your JSON Output 2
{{
  "A": "A person is building muscle and strength.",
  "B": "A person is practicing for a marathon.",
  "C": "A person is training for a cycling race.",
  "D": "A person is preparing for a swimming competition."
}}

### Your Task
Question: {question}
A: {option_A}
B: {option_B}
C: {option_C}
D: {option_D}

### Your JSON Output
"""

# --------------------------------------------------------------------------
# --- 4. LLM 호출 함수 정의 ---
# --------------------------------------------------------------------------
def preprocess_options_for_clip(row, llm_pipeline, prompt_template):
    """LLM을 사용하여 한 행의 보기 4개를 CLIP 친화적인 캡션으로 변환합니다."""
    prompt = prompt_template.format(
        question=row['Question'],
        option_A=row['A'], option_B=row['B'], option_C=row['C'], option_D=row['D']
    )
    # 파이프라인에 맞는 메시지 형식 생성
    messages = [{"role": "user", "content": prompt}]
    
    try:
        outputs = llm_pipeline(messages, max_new_tokens=256, do_sample=False, temperature=0.1)
        # 파이프라인 출력에서 마지막 응답(assistant의 답변) 텍스트 추출
        response_content = outputs[0]["generated_text"][-1]['content']
        
        # LLM 응답에서 JSON 부분만 정확히 추출
        json_match = re.search(r'\{.*\}', response_content, re.DOTALL)
        
        if json_match:
            processed_options = json.loads(json_match.group(0))
            # 딕셔너리의 값(A, B, C, D에 해당하는 캡션)을 리스트로 반환
            return list(processed_options.values())
        else:
            # JSON 파싱 실패 시 경고 출력 및 원본 보기 반환
            print(f"\nWarning: 행 ID {row.get('ID', 'N/A')}의 JSON 파싱 실패. 원본 보기를 사용합니다. Raw: '{response_content}'")
            return [row['A'], row['B'], row['C'], row['D']]

    except Exception as e:
        # 그 외 예외 발생 시 에러 출력 및 원본 보기 반환
        print(f"\nError: ID {row.get('ID', 'N/A')} 처리 중 예외 발생: {e}")
        return [row['A'], row['B'], row['C'], row['D']]

# --------------------------------------------------------------------------
# --- 5. 메인 실행 루프 ---
# --------------------------------------------------------------------------
try:
    df = pd.read_csv(input_csv_path)
    print(f"\n'{input_csv_path}'에서 {len(df)}개의 행을 읽었습니다. 전처리를 시작합니다.")
except FileNotFoundError:
    print(f"❌ 입력 파일 '{input_csv_path}'을(를) 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 결과를 저장할 리스트 초기화
processed_rows = []

# tqdm을 사용하여 전체 진행 상황 표시
for index, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Options for CLIP"):
    # LLM 호출을 한 번으로 줄여 4개의 보기를 모두 변환
    rewritten_options = preprocess_options_for_clip(row, llm_pipe, prompt_template_for_clip)
    
    # 원본 데이터에 변환된 보기 추가
    new_row = row.to_dict()
    if len(rewritten_options) == 4:
        new_row['caption_A'] = rewritten_options[0]
        new_row['caption_B'] = rewritten_options[1]
        new_row['caption_C'] = rewritten_options[2]
        new_row['caption_D'] = rewritten_options[3]
    processed_rows.append(new_row)

# --------------------------------------------------------------------------
# --- 6. 결과 종합 및 저장 ---
# --------------------------------------------------------------------------
print("\n전처리가 완료되었습니다. 결과를 CSV 파일로 저장합니다.")

# 변환된 결과로 새로운 DataFrame 생성
final_df = pd.DataFrame(processed_rows)

# 결과를 새로운 CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig 인코딩)
final_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업 완료! 최종 결과가 '{output_csv_path}' 파일에 저장되었습니다.")
print("\n--- 최종 전처리 결과 미리보기 (상위 5개) ---")
# 보기 좋게 일부 열만 선택하여 출력
preview_columns = ['ID', 'Question', 'caption_A', 'caption_B', 'caption_C', 'caption_D']
print(final_df[preview_columns].head())

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM (Llama 3.1)을 로딩합니다...


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.86s/it]
Device set to use cuda:0


✅ LLM 로딩이 완료되었습니다.
입력 파일: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv
출력 파일: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_for_clip.csv

'/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'에서 60개의 행을 읽었습니다. 전처리를 시작합니다.


Preprocessing Options for CLIP:   0%|          | 0/60 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Preprocessing Options for CLIP:   2%|▏         | 1/60 [00:01<01:56,  1.97s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Preprocessing Options for CLIP:   3%|▎         | 2/60 [00:03<01:24,  1.46s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Preprocessing Options for CLIP:   5%|▌         | 3/60 [00:04<01:11,  1.26s/it]The following generation flags are not valid and


전처리가 완료되었습니다. 결과를 CSV 파일로 저장합니다.

✅ 모든 작업 완료! 최종 결과가 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_for_clip.csv' 파일에 저장되었습니다.

--- 최종 전처리 결과 미리보기 (상위 5개) ---
         ID                                           Question  \
0  TEST_000  What might be the purpose of the person's work...   
1  TEST_001          What color is the shoe rack in the image?   
2  TEST_002    What might be the deer's behavior in the image?   
3  TEST_003  Considering the equipment used in the image, w...   
4  TEST_004  What is a common ingredient in Gimbap (Korean ...   

                                   caption_A  \
0  A person is building muscle and strength.   
1                    The shoe rack is black.   
2                        The deer is eating.   
3                  This is a Pilates studio.   
4              There is cheese in the photo.   

                                caption_B  \
0  A person is practicing for a marathon.   
1             